# 🧠 Multi-Model Code Converter and Runner

This project is an AI-powered **code conversion tool** that converts **Python code into multiple programming languages** using modern Large Language Models (LLMs). It provides an interactive **Gradio web interface** where users can select a model, choose a target language, convert Python code, and execute the generated program.

The system supports both **frontier models** and **open-source models**, and gracefully skips models whose API keys are not available.

---

# 🚀 Features

* Convert **Python → C++**
* Convert **Python → Rust**
* Convert **Python → Go**
* Convert **Python → Java**
* Run compiled code automatically after conversion
* Choose between **multiple LLM models**
* Interactive **Gradio UI**
* Automatically opens in your browser
* Supports both **API-based models and local models**

---

# 🧠 Supported Models

The project integrates models from multiple providers.

### Frontier Models

* GPT-5 (OpenAI)
* Claude Sonnet (Anthropic)
* Grok-4 (xAI / Groq)
* Gemini 2.5 Pro (Google)

### Open Source / Open Weight Models

* DeepSeek-Coder-V2
* Qwen2.5-Coder
* GPT-OSS-20B
* Qwen3-Coder

Local models can be run using **Ollama**.

---

# 🖥 Supported Target Languages

| Language | Compilation Command        | Execution Command |
| -------- | -------------------------- | ----------------- |
| C++      | `g++ output.cpp -o output` | `./output`        |
| Rust     | `rustc output.rs`          | `./output`        |
| Go       | `go build output.go`       | `./output`        |
| Java     | `javac Main.java`          | `java Main`       |

---

# 📦 Requirements

Install Python dependencies:

```bash
pip install gradio openai anthropic groq google-generativeai ollama requests python-dotenv
```

You must also install compilers depending on the language you want to run.

### C++

Install `g++`

### Rust

Install Rust:

```bash
rustup install stable
```

### Go

Install Go:

```bash
sudo apt install golang
```

### Java

Install JDK 17+:

```bash
sudo apt install openjdk-17-jdk
```

---

# 🔑 API Keys (Optional)

The project works even if some APIs are missing.

Set environment variables:

```
OPENAI_API_KEY=your_key
ANTHROPIC_API_KEY=your_key
GROQ_API_KEY=your_key
GOOGLE_API_KEY=your_key
OPENROUTER_API_KEY=your_key
```

If a key is missing, that model will automatically be skipped.

---

# ▶️ Running the Application

Start the application:

```bash
python app.py
```

The interface will automatically open in your browser:

```
http://127.0.0.1:7860
```

---

# 🖥 Interface Overview

The web interface contains:

* **Model Selector** – choose which LLM performs the conversion
* **Target Language Selector** – choose C++, Rust, Go, or Java
* **Python Code Editor** – paste or write Python code
* **Convert + Run Button** – converts and executes the program
* **Converted Code Panel** – displays generated code
* **Program Output Panel** – shows execution results

---

# 📌 Example

### Python Input

```python
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n-1)

print(factorial(5))
```

### Generated C++ Code

```cpp
#include <iostream>
using namespace std;

int factorial(int n){
    if(n==0) return 1;
    return n * factorial(n-1);
}

int main(){
    cout << factorial(5);
}
```

### Program Output

```
120
```

---

# 📂 Project Structure

```
project/
│
├── app.py
├── README.md
├── output.cpp
├── output.rs
├── output.go
└── Main.java
```

---

# ⚠️ Notes

* Some models require API keys.
* Local models require **Ollama**.
* Java programs must use a `Main` class.
* Compilation errors will appear in the **Program Output** panel.

---

# 🔮 Future Improvements

* Side-by-side code comparison
* Multi-model benchmarking
* Performance measurement
* Support for additional languages (CUDA, C#, Kotlin)
* Code optimization suggestions

---

# 👨‍💻 Author

Developed as part of an **AI / LLM experimentation project** exploring how large language models can assist in cross-language code generation and execution.


In [1]:
import gradio as gr
import os
import subprocess
import ollama
import requests
from openai import OpenAI

In [2]:
#API keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")

In [3]:
# =========================
# CLIENTS
# =========================

openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY and anthropic else None
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY and Groq else None

if GOOGLE_API_KEY and genai:
    genai.configure(api_key=GOOGLE_API_KEY)

In [4]:

# =========================
# DIFFERENT MODELS THAT YOU CAN CHOOSE TO RUN
# =========================

MODELS = {
    "gpt-5": "openai",
    "claude-sonnet": "anthropic",
    "grok-4": "groq",
    "gemini-2.5-pro": "google",
    "qwen2.5-coder": "ollama",
    "deepseek-coder-v2": "ollama",
    "gpt-oss:20b": "ollama",
    "qwen/qwen3-coder-30b-a3b-instruct": "openrouter",
}

In [5]:

# =========================
# LANGUAGE CONFIG
# =========================

LANGUAGES = {
    "cpp": {
        "compile": ["g++", "output.cpp", "-o", "output"],
        "run": ["./output"]
    },
    "rust": {
        "compile": ["rustc", "output.rs"],
        "run": ["./output"]
    },
    "go": {
        "compile": ["go", "build", "output.go"],
        "run": ["./output"]
    },
    "java": {
        "file": "Main.java",
        "compile": ["javac", "Main.java"],
        "run": ["java", "Main"]
    }
}


In [6]:
# =========================
# PROMPT
# =========================

def build_prompt(code, language):

    return f"""
Convert the following Python code to {language}.
Return only the code.

Python Code:
{code}
"""

In [7]:
# =========================
# MODEL ROUTER
# =========================

def generate_code(model, python_code, language):

    provider = MODELS[model]
    prompt = build_prompt(python_code, language)

    try:

        if provider == "openai" and openai_client:

            response = openai_client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":prompt}]
            )

            return response.choices[0].message.content


        elif provider == "anthropic" and anthropic_client:

            response = anthropic_client.messages.create(
                model=model,
                max_tokens=2000,
                messages=[{"role":"user","content":prompt}]
            )

            return response.content[0].text


        elif provider == "groq" and groq_client:

            response = groq_client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":prompt}]
            )

            return response.choices[0].message.content


        elif provider == "google" and GOOGLE_API_KEY:

            model_g = genai.GenerativeModel(model)
            response = model_g.generate_content(prompt)

            return response.text


        elif provider == "ollama":

            response = ollama.chat(
                model=model,
                messages=[{"role":"user","content":prompt}]
            )

            return response['message']['content']


        elif provider == "openrouter" and OPENROUTER_KEY:

            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {OPENROUTER_KEY}"
                },
                json={
                    "model": model,
                    "messages":[{"role":"user","content":prompt}]
                }
            )

            return response.json()["choices"][0]["message"]["content"]

        else:
            return "⚠️ Model unavailable (missing API key)"

    except Exception as e:
        return f"Error: {e}"


In [8]:

# =========================
# SAVE CODE
# =========================

def save_code(code, extension):

    code = code.replace("```","").strip()

    filename = f"output.{extension}"

    with open(filename,"w") as f:
        f.write(code)

    return filename

In [9]:
# =========================
# COMPILE + RUN
# =========================

def compile_and_run(lang):

    try:

        subprocess.run(LANGUAGES[lang]["compile"], check=True)

        result = subprocess.run(
            LANGUAGES[lang]["run"],
            capture_output=True,
            text=True
        )

        return result.stdout

    except Exception as e:
        return str(e)

In [10]:
# =========================
# MAIN FUNCTION
# =========================

def convert_and_run(model, language, python_code):

    generated = generate_code(model, python_code, language)

    if "⚠️" in generated:
        return generated, ""

    save_code(generated, language)

    output = compile_and_run(language)

    return generated, output

In [11]:
# =========================
# GRADIO UI
# =========================

with gr.Blocks() as demo:

    gr.Markdown("# 🧠 Multi-Model Code Converter")

    with gr.Row():

        model = gr.Dropdown(
            list(MODELS.keys()),
            label="Select Model",
            value="gpt-5"
        )

        language = gr.Dropdown(
            list(LANGUAGES.keys()),
            label="Target Language",
            value="cpp"
        )

    python_code = gr.Code(
        label="Python Code",
        language="python",
        value="""
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""
    )

    convert_btn = gr.Button("Convert + Run")

    converted_code = gr.Code(label="Converted Code")

    program_output = gr.Textbox(label="Program Output")

    convert_btn.click(
        convert_and_run,
        inputs=[model, language, python_code],
        outputs=[converted_code, program_output]
    )


demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
